AutoGluon - Predicción de ventas (tn) por producto para febrero 2020

In [1]:
# 📦 1. Importar librerías
import pandas as pd

In [2]:
# 💬 Instalar AutoGluon si es necesario
%pip install autogluon.timeseries

from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

Note: you may need to restart the kernel to use updated packages.


c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 📄 2. Cargar datasets
import os

drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Data/'
filename = 'sell-in.txt'
filepath = os.path.join(drive_base_path, filename)
df_sellin = pd.read_csv(filepath, sep='\t')
print(df_sellin.head(10))

filename = 'tb_productos.txt'
filepath = os.path.join(drive_base_path, filename)
df_productos = pd.read_csv(filepath, sep='\t')

filename = 'product_id_apredecir201912.txt'
filepath = os.path.join(drive_base_path, filename)
df_a_predecir = pd.read_csv(filepath, sep='\t')

   periodo  customer_id  product_id  plan_precios_cuidados  cust_request_qty  \
0   201701        10234       20524                      0                 2   
1   201701        10032       20524                      0                 1   
2   201701        10217       20524                      0                 1   
3   201701        10125       20524                      0                 1   
4   201701        10012       20524                      0                11   
5   201701        10080       20524                      0                 1   
6   201701        10015       20524                      0                 4   
7   201701        10062       20524                      0                 1   
8   201701        10159       20524                      0                 3   
9   201701        10183       20524                      0                 1   

   cust_request_tn       tn  
0          0.05300  0.05300  
1          0.13628  0.13628  
2          0.03028  0.03028  

In [4]:
# 📄 Leer lista de productos a predecir
with open(filepath, "r") as f:
    product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]
    
print(product_ids)

[20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010, 20011, 20012, 20013, 20014, 20015, 20016, 20017, 20018, 20019, 20020, 20021, 20022, 20023, 20024, 20025, 20026, 20027, 20028, 20029, 20030, 20031, 20032, 20033, 20035, 20037, 20038, 20039, 20041, 20042, 20043, 20044, 20045, 20046, 20047, 20049, 20050, 20051, 20052, 20053, 20054, 20055, 20056, 20057, 20058, 20059, 20061, 20062, 20063, 20065, 20066, 20067, 20068, 20069, 20070, 20071, 20072, 20073, 20074, 20075, 20076, 20077, 20079, 20080, 20081, 20082, 20084, 20085, 20086, 20087, 20089, 20090, 20091, 20092, 20093, 20094, 20095, 20096, 20097, 20099, 20100, 20101, 20102, 20103, 20106, 20107, 20108, 20109, 20111, 20112, 20114, 20116, 20117, 20118, 20119, 20120, 20121, 20122, 20123, 20124, 20125, 20126, 20127, 20129, 20130, 20132, 20133, 20134, 20135, 20137, 20138, 20139, 20140, 20142, 20143, 20144, 20145, 20146, 20148, 20150, 20151, 20152, 20153, 20155, 20157, 20158, 20159, 20160, 20161, 20162, 20164, 20166, 20167, 20168

In [5]:
# 🧹 3. Preprocesamiento
# Convertir periodo a datetime
df_sellin['timestamp'] = pd.to_datetime(df_sellin['periodo'], format='%Y%m')

In [6]:
# Filtrar hasta dic 2019 y productos requeridos
df_filtered = df_sellin[
    (df_sellin['timestamp'] <= '2019-12-01') &
    (df_sellin['product_id'].isin(product_ids))
]

In [7]:
# Agregar tn por periodo, cliente y producto
df_grouped = df_filtered.groupby(['timestamp', 'customer_id', 'product_id'], as_index=False)['tn'].sum()

In [8]:
# Agregar tn total por periodo y producto
df_monthly_product = df_grouped.groupby(['timestamp', 'product_id'], as_index=False)['tn'].sum()

In [9]:
# Agregar columna 'item_id' para AutoGluon
df_monthly_product['item_id'] = df_monthly_product['product_id']

In [10]:
# ⏰ 4. Crear TimeSeriesDataFrame
ts_data = TimeSeriesDataFrame.from_data_frame(
    df_monthly_product,
    id_column='item_id',
    timestamp_column='timestamp'
)

In [11]:
# Completar valores faltantes
ts_data = ts_data.fill_missing_values()

In [ ]:
# ⚙️ 5. Definir y entrenar predictor
predictor = TimeSeriesPredictor(
    prediction_length=2,
    target='tn',
    freq='MS'  # Frecuencia mensual (Month Start), 
)

predictor.fit(ts_data, num_val_windows=2, time_limit=60*60)

Beginning AutoGluon training... Time limit = 7200s
AutoGluon will save models to 'c:\Repositorios-Ing.Carlos-Cicconi\labo3-2025r\src\AutoGluon\AutogluonModels\ag-20250729_003839'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.4
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          4
GPU Count:          0
Memory Avail:       1.40 GB / 7.84 GB (17.9%)
Disk Space Avail:   22.51 GB / 237.85 GB (9.5%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 4,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'time_limit': 7200,
 'verbosity': 2}

train_data with frequency 'IRREG' has been resampled to frequency 'MS

In [13]:
# 🔮 6. Generar predicción
forecast = predictor.predict(ts_data)

data with frequency 'IRREG' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


In [14]:
# Extraer predicción media
forecast_mean = forecast['mean'].reset_index()
print(forecast_mean.columns)

Index(['item_id', 'timestamp', 'mean'], dtype='object')


In [15]:
# Tomar solo item_id y la predicción 'mean'
resultado = forecast['mean'].reset_index()[['item_id', 'mean']]
resultado.columns = ['product_id', 'tn']

# Filtrar solo febrero 2020
resultado = forecast['mean'].reset_index()
resultado = resultado[resultado['timestamp'] == '2020-02-01']

# Renombrar columnas
resultado = resultado[['item_id', 'mean']]
resultado.columns = ['product_id', 'tn']


In [ ]:
# 💾 7. Guardar archivo
resultado.to_csv("C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/AutoGluon/predicciones_febrero2020_nvw2.csv", index=False)
resultado.head()

,product_id,tn
1,20001,1270.154676
3,20002,1004.235828
5,20003,614.978656
7,20004,475.119935
9,20005,502.771489


## 🔧 Usar predicciones AutoGluon como Features

Las predicciones de AutoGluon pueden usarse como features de diferentes maneras:

1. **Feature directa**: Usar la predicción como variable predictora
2. **Feature de tendencia**: Calcular diferencias con valores históricos  
3. **Feature de confianza**: Usar intervalos de predicción como medida de incertidumbre
4. **Feature combinada**: Combinar con otras predicciones (ensemble)
5. **Feature temporal**: Crear patrones temporales basados en predicciones

In [ ]:
# 🔄 8. Preparar dataset base para agregar features
# Cargar datos históricos para crear features combinadas

# Crear dataset con datos hasta enero 2020 (para predecir febrero)
df_feature_base = df_sellin[
    (df_sellin['timestamp'] <= '2020-01-01') &
    (df_sellin['product_id'].isin(product_ids))
].copy()

# Agregar por producto y período
df_historical = df_feature_base.groupby(['timestamp', 'product_id'], as_index=False)['tn'].sum()

print("=== DATASET BASE PARA FEATURES ===")
print(f"Períodos disponibles: {df_historical['timestamp'].min()} a {df_historical['timestamp'].max()}")
print(f"Productos únicos: {df_historical['product_id'].nunique()}")
print(f"Total registros: {len(df_historical)}")

# Mostrar últimos datos por producto
print("\n=== ÚLTIMOS DATOS POR PRODUCTO ===")
ultimos_datos = df_historical[df_historical['timestamp'] == '2020-01-01']
print(ultimos_datos.head())

In [ ]:
# 🎯 MÉTODO 1: Feature directa - Predicción AutoGluon como variable predictora
print("=== MÉTODO 1: FEATURE DIRECTA ===")

# Crear dataset con última observación histórica + predicción AutoGluon
df_method1 = ultimos_datos[['product_id', 'tn']].copy()
df_method1 = df_method1.rename(columns={'tn': 'tn_enero_2020'})

# Agregar predicción AutoGluon como feature
df_method1 = df_method1.merge(resultado[['product_id', 'tn']], on='product_id', how='left')
df_method1 = df_method1.rename(columns={'tn': 'autogluon_pred_feb2020'})

# Crear features adicionales basadas en la predicción
df_method1['pred_vs_ultimo'] = df_method1['autogluon_pred_feb2020'] / df_method1['tn_enero_2020']
df_method1['pred_vs_ultimo_dif'] = df_method1['autogluon_pred_feb2020'] - df_method1['tn_enero_2020']

print("Dataset con feature directa:")
print(df_method1.head())
print(f"\nProductos con predicción: {df_method1['autogluon_pred_feb2020'].notna().sum()}")

# Estadísticas de la nueva feature
print(f"\n=== ESTADÍSTICAS FEATURE AUTOGLUON ===")
print(f"Media predicción: {df_method1['autogluon_pred_feb2020'].mean():.2f}")
print(f"Mediana predicción: {df_method1['autogluon_pred_feb2020'].median():.2f}")
print(f"Min predicción: {df_method1['autogluon_pred_feb2020'].min():.2f}")
print(f"Max predicción: {df_method1['autogluon_pred_feb2020'].max():.2f}")
print(f"Ratio promedio pred/último: {df_method1['pred_vs_ultimo'].mean():.3f}")

In [ ]:
# 📈 MÉTODO 2: Features de tendencia - Comparar predicción con patrones históricos
print("=== MÉTODO 2: FEATURES DE TENDENCIA ===")

# Calcular estadísticas históricas por producto (últimos 6 meses)
df_last_6m = df_historical[df_historical['timestamp'] >= '2019-08-01'].copy()
stats_historicas = df_last_6m.groupby('product_id')['tn'].agg({
    'promedio_6m': 'mean',
    'mediana_6m': 'median', 
    'std_6m': 'std',
    'min_6m': 'min',
    'max_6m': 'max',
    'tendencia_6m': lambda x: (x.iloc[-1] - x.iloc[0]) / len(x) if len(x) > 1 else 0
}).reset_index()

# Combinar con predicciones AutoGluon
df_method2 = stats_historicas.merge(resultado[['product_id', 'tn']], on='product_id', how='left')
df_method2 = df_method2.rename(columns={'tn': 'autogluon_pred_feb2020'})

# Crear features de tendencia
df_method2['pred_vs_promedio_6m'] = df_method2['autogluon_pred_feb2020'] / df_method2['promedio_6m']
df_method2['pred_vs_mediana_6m'] = df_method2['autogluon_pred_feb2020'] / df_method2['mediana_6m']
df_method2['pred_desviacion_std'] = (df_method2['autogluon_pred_feb2020'] - df_method2['promedio_6m']) / df_method2['std_6m']

# Feature de anomalía (si la predicción está fuera del rango histórico)
df_method2['pred_fuera_rango'] = (
    (df_method2['autogluon_pred_feb2020'] < df_method2['min_6m']) | 
    (df_method2['autogluon_pred_feb2020'] > df_method2['max_6m'])
).astype(int)

# Feature de alineación con tendencia
df_method2['pred_alineada_tendencia'] = (
    df_method2['autogluon_pred_feb2020'] > df_method2['promedio_6m']
).astype(int) == (df_method2['tendencia_6m'] > 0).astype(int)

print("Dataset con features de tendencia:")
print(df_method2[['product_id', 'autogluon_pred_feb2020', 'pred_vs_promedio_6m', 
                  'pred_desviacion_std', 'pred_fuera_rango', 'pred_alineada_tendencia']].head())

print(f"\n=== RESUMEN FEATURES DE TENDENCIA ===")
print(f"Productos con predicción > promedio: {(df_method2['pred_vs_promedio_6m'] > 1).sum()}")
print(f"Productos con predicción fuera de rango: {df_method2['pred_fuera_rango'].sum()}")
print(f"Productos con predicción alineada a tendencia: {df_method2['pred_alineada_tendencia'].sum()}")

In [ ]:
# ⏰ MÉTODO 3: Features temporales - Crear lags y patrones temporales
print("=== MÉTODO 3: FEATURES TEMPORALES ===")

# Crear dataset temporal completo con predicción incorporada
df_temporal = df_historical.copy()

# Agregar la predicción como período futuro
pred_temporal = resultado.copy()
pred_temporal['timestamp'] = pd.to_datetime('2020-02-01')
pred_temporal = pred_temporal.rename(columns={'tn': 'tn'})

# Combinar datos históricos con predicción
df_temporal_completo = pd.concat([df_temporal, pred_temporal], ignore_index=True)
df_temporal_completo = df_temporal_completo.sort_values(['product_id', 'timestamp'])

# Crear features de lags usando la predicción
df_method3 = df_temporal_completo.groupby('product_id').apply(
    lambda x: x.assign(
        tn_lag_1=x['tn'].shift(1),
        tn_lag_2=x['tn'].shift(2),
        tn_lag_3=x['tn'].shift(3),
        tn_rolling_3m=x['tn'].rolling(3, min_periods=1).mean(),
        tn_rolling_6m=x['tn'].rolling(6, min_periods=1).mean(),
        tn_trend_3m=x['tn'].rolling(3, min_periods=2).apply(
            lambda y: (y.iloc[-1] - y.iloc[0]) / (len(y) - 1) if len(y) > 1 else 0
        )
    )
).reset_index(drop=True)

# Filtrar solo febrero 2020 (donde queremos usar las features)
df_method3_feb = df_method3[df_method3['timestamp'] == '2020-02-01'].copy()

# Crear features adicionales específicas
df_method3_feb['pred_vs_lag1'] = df_method3_feb['tn'] / df_method3_feb['tn_lag_1']
df_method3_feb['pred_vs_rolling3m'] = df_method3_feb['tn'] / df_method3_feb['tn_rolling_3m']
df_method3_feb['pred_vs_rolling6m'] = df_method3_feb['tn'] / df_method3_feb['tn_rolling_6m']
df_method3_feb['pred_aceleración'] = df_method3_feb['tn'] - 2*df_method3_feb['tn_lag_1'] + df_method3_feb['tn_lag_2']

print("Dataset con features temporales:")
print(df_method3_feb[['product_id', 'tn', 'tn_lag_1', 'tn_rolling_3m', 
                      'pred_vs_lag1', 'pred_vs_rolling3m', 'pred_aceleración']].head())

print(f"\n=== ESTADÍSTICAS FEATURES TEMPORALES ===")
print(f"Ratio promedio pred/lag1: {df_method3_feb['pred_vs_lag1'].mean():.3f}")
print(f"Ratio promedio pred/rolling3m: {df_method3_feb['pred_vs_rolling3m'].mean():.3f}")
print(f"Aceleración promedio: {df_method3_feb['pred_aceleración'].mean():.2f}")

In [ ]:
# 🤝 MÉTODO 4: Features de ensemble - Combinar con otras predicciones
print("=== MÉTODO 4: FEATURES DE ENSEMBLE ===")

# Crear predicciones simples como baseline para comparación
# Predicción naive: último valor conocido (enero 2020)
pred_naive = ultimos_datos[['product_id', 'tn']].copy()
pred_naive = pred_naive.rename(columns={'tn': 'pred_naive'})

# Predicción promedio móvil 3 meses
pred_ma3 = df_last_6m.groupby('product_id')['tn'].tail(3).groupby('product_id').mean().reset_index()
pred_ma3 = pred_ma3.rename(columns={'tn': 'pred_ma3'})

# Predicción de tendencia lineal simple
def calcular_tendencia_lineal(group):
    if len(group) < 2:
        return group['tn'].iloc[-1]
    x = range(len(group))
    y = group['tn'].values
    # Regresión lineal simple: y = mx + b
    n = len(x)
    m = (n * sum(x[i] * y[i] for i in range(n)) - sum(x) * sum(y)) / (n * sum(x_i**2 for x_i in x) - sum(x)**2)
    b = (sum(y) - m * sum(x)) / n
    # Predecir siguiente período
    return m * len(group) + b

pred_trend = df_last_6m.groupby('product_id').apply(calcular_tendencia_lineal).reset_index()
pred_trend.columns = ['product_id', 'pred_trend']

# Combinar todas las predicciones
df_method4 = resultado[['product_id', 'tn']].copy()
df_method4 = df_method4.rename(columns={'tn': 'pred_autogluon'})
df_method4 = df_method4.merge(pred_naive, on='product_id', how='left')
df_method4 = df_method4.merge(pred_ma3, on='product_id', how='left')
df_method4 = df_method4.merge(pred_trend, on='product_id', how='left')

# Crear features de ensemble
df_method4['ensemble_promedio'] = df_method4[['pred_autogluon', 'pred_naive', 'pred_ma3', 'pred_trend']].mean(axis=1)
df_method4['ensemble_mediana'] = df_method4[['pred_autogluon', 'pred_naive', 'pred_ma3', 'pred_trend']].median(axis=1)

# Features de consenso/dispersión
df_method4['pred_std'] = df_method4[['pred_autogluon', 'pred_naive', 'pred_ma3', 'pred_trend']].std(axis=1)
df_method4['pred_cv'] = df_method4['pred_std'] / df_method4['ensemble_promedio']  # Coeficiente de variación
df_method4['pred_range'] = (df_method4[['pred_autogluon', 'pred_naive', 'pred_ma3', 'pred_trend']].max(axis=1) - 
                           df_method4[['pred_autogluon', 'pred_naive', 'pred_ma3', 'pred_trend']].min(axis=1))

# Features de ranking/posición
df_method4['autogluon_rank'] = df_method4[['pred_autogluon', 'pred_naive', 'pred_ma3', 'pred_trend']].rank(axis=1)['pred_autogluon']
df_method4['autogluon_es_max'] = (df_method4['pred_autogluon'] == 
                                 df_method4[['pred_autogluon', 'pred_naive', 'pred_ma3', 'pred_trend']].max(axis=1)).astype(int)
df_method4['autogluon_es_min'] = (df_method4['pred_autogluon'] == 
                                 df_method4[['pred_autogluon', 'pred_naive', 'pred_ma3', 'pred_trend']].min(axis=1)).astype(int)

print("Dataset con features de ensemble:")
print(df_method4[['product_id', 'pred_autogluon', 'ensemble_promedio', 'pred_std', 
                  'pred_cv', 'autogluon_rank', 'autogluon_es_max']].head())

print(f"\n=== ESTADÍSTICAS ENSEMBLE ===")
print(f"Dispersión promedio (std): {df_method4['pred_std'].mean():.2f}")
print(f"CV promedio: {df_method4['pred_cv'].mean():.3f}")
print(f"AutoGluon es máximo en: {df_method4['autogluon_es_max'].sum()} productos")
print(f"AutoGluon es mínimo en: {df_method4['autogluon_es_min'].sum()} productos")

In [ ]:
# 📊 EXPORTAR Y RESUMIR todos los enfoques de features
print("=== EXPORTACIÓN DE DATASETS CON FEATURES ===")

# Crear directorio de salida
output_dir = "C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/AutoGluon"
os.makedirs(output_dir, exist_ok=True)

# Exportar cada método
datasets_exportar = [
    (df_method1, "features_metodo1_directa.csv", "Feature directa"),
    (df_method2, "features_metodo2_tendencia.csv", "Features de tendencia"),
    (df_method3_feb, "features_metodo3_temporal.csv", "Features temporales"),
    (df_method4, "features_metodo4_ensemble.csv", "Features de ensemble")
]

for df, filename, descripcion in datasets_exportar:
    filepath = os.path.join(output_dir, filename)
    df.to_csv(filepath, index=False)
    print(f"✅ {descripcion}: {filepath}")
    print(f"   Filas: {len(df)}, Columnas: {len(df.columns)}")

print(f"\n=== RESUMEN Y RECOMENDACIONES ===")
print(f"🎯 CASOS DE USO para cada método:")
print(f"")
print(f"📌 MÉTODO 1 (Feature directa):")
print(f"   - Usar cuando: Quieres la predicción como input directo")
print(f"   - Ideal para: Modelos que pueden manejar correlación alta")
print(f"   - Features clave: 'autogluon_pred_feb2020', 'pred_vs_ultimo'")
print(f"")
print(f"📈 MÉTODO 2 (Features de tendencia):")
print(f"   - Usar cuando: Quieres capturar patrones vs histórico")
print(f"   - Ideal para: Detectar anomalías o cambios de patrón")
print(f"   - Features clave: 'pred_vs_promedio_6m', 'pred_desviacion_std', 'pred_fuera_rango'")
print(f"")
print(f"⏰ MÉTODO 3 (Features temporales):")
print(f"   - Usar cuando: Quieres incorporar dinámicas temporales")
print(f"   - Ideal para: Series temporales con patrones estacionales")
print(f"   - Features clave: 'pred_vs_lag1', 'pred_vs_rolling3m', 'pred_aceleración'")
print(f"")
print(f"🤝 MÉTODO 4 (Features de ensemble):")
print(f"   - Usar cuando: Quieres robustecer con múltiples predicciones")
print(f"   - Ideal para: Reducir riesgo de sobreajuste a un solo modelo")
print(f"   - Features clave: 'ensemble_promedio', 'pred_cv', 'autogluon_rank'")
print(f"")
print(f"🏆 RECOMENDACIÓN GENERAL:")
print(f"   1. Comenzar con MÉTODO 4 (ensemble) por su robustez")
print(f"   2. Combinar con MÉTODO 2 (tendencia) para detectar anomalías")
print(f"   3. Usar MÉTODO 3 (temporal) si hay estacionalidad clara")
print(f"   4. MÉTODO 1 solo si el modelo final puede manejar high correlation")
print(f"")
print(f"💡 PRÓXIMOS PASOS:")
print(f"   - Probar cada conjunto de features en tu modelo final")
print(f"   - Usar validación cruzada para comparar rendimiento")
print(f"   - Considerar combinaciones de métodos según el caso")

## 🕒 Dynamic Time Warping (DTW) para predicciones

DTW puede mejorar las predicciones mediante:

1. **Encontrar productos similares** con patrones desalineados temporalmente
2. **Manejo de estacionalidades irregulares** (patrones "estirados" o "comprimidos")
3. **Robustez ante datos faltantes** o históricos de diferente longitud
4. **Identificación de templates** - productos con patrones estables como referencia
5. **Predicción por analogía** - usar patrones similares históricos para predecir

In [ ]:
# 🔧 INSTALACIÓN Y CONFIGURACIÓN DTW
print("=== CONFIGURANDO DYNAMIC TIME WARPING ===")

# Instalar dtaidistance si es necesario
try:
    from dtaidistance import dtw
    from dtaidistance.dtw import distance_matrix_fast
    print("✅ dtaidistance ya instalado")
except ImportError:
    print("📦 Instalando dtaidistance...")
    %pip install dtaidistance
    from dtaidistance import dtw
    from dtaidistance.dtw import distance_matrix_fast

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt

print("✅ Librerías DTW importadas correctamente")

In [ ]:
# 📊 PREPARAR DATOS PARA ANÁLISIS DTW
print("=== PREPARACIÓN DE SERIES TEMPORALES PARA DTW ===")

# Crear matriz de series temporales por producto (últimos 12 meses)
df_dtw_base = df_historical[df_historical['timestamp'] >= '2019-02-01'].copy()
df_dtw_base = df_dtw_base.sort_values(['product_id', 'timestamp'])

# Crear pivot table: productos vs períodos
ts_matrix = df_dtw_base.pivot(index='product_id', columns='timestamp', values='tn')
ts_matrix = ts_matrix.fillna(0)  # Rellenar faltantes con 0

print(f"Matriz de series temporales: {ts_matrix.shape}")
print(f"Products: {ts_matrix.shape[0]}, Períodos: {ts_matrix.shape[1]}")
print(f"Períodos disponibles: {ts_matrix.columns.min()} a {ts_matrix.columns.max()}")

# Mostrar algunas series
print(f"\n=== PRIMERAS SERIES TEMPORALES ===")
print(ts_matrix.head())

# Normalizar series temporales para DTW (importante para comparación justa)
scaler_dtw = MinMaxScaler()
ts_matrix_norm = pd.DataFrame(
    scaler_dtw.fit_transform(ts_matrix.T).T,
    index=ts_matrix.index,
    columns=ts_matrix.columns
)

print(f"\n=== ESTADÍSTICAS POST-NORMALIZACIÓN ===")
print(f"Min global: {ts_matrix_norm.min().min():.3f}")
print(f"Max global: {ts_matrix_norm.max().max():.3f}")
print(f"Series con varianza > 0: {(ts_matrix_norm.var(axis=1) > 0.01).sum()}")

# Filtrar productos con suficiente variabilidad
productos_activos = ts_matrix_norm[ts_matrix_norm.var(axis=1) > 0.01].index.tolist()
print(f"Productos activos para DTW: {len(productos_activos)}")

ts_matrix_active = ts_matrix_norm.loc[productos_activos]

In [ ]:
# 🔄 CALCULAR MATRIZ DE DISTANCIAS DTW
print("=== CALCULANDO DISTANCIAS DTW ===")

# Convertir a numpy arrays para DTW
series_arrays = [ts_matrix_active.iloc[i].values for i in range(len(ts_matrix_active))]

# Calcular matriz de distancias DTW (puede tomar tiempo con muchos productos)
print(f"Calculando DTW para {len(series_arrays)} productos...")

# Para eficiencia, limitamos a los primeros N productos si hay demasiados
max_products = min(50, len(series_arrays))  # Limitar para demo
if len(series_arrays) > max_products:
    print(f"⚠️  Limitando análisis a {max_products} productos por eficiencia")
    series_arrays = series_arrays[:max_products]
    productos_dtw = productos_activos[:max_products]
else:
    productos_dtw = productos_activos

# Calcular matriz de distancias DTW
print("Calculando matriz de distancias... (puede tomar unos minutos)")
try:
    distance_matrix = distance_matrix_fast(series_arrays)
    print(f"✅ Matriz DTW calculada: {distance_matrix.shape}")
except Exception as e:
    print(f"⚠️  Error con función rápida, usando método estándar: {e}")
    # Fallback a método manual
    n = len(series_arrays)
    distance_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            dist = dtw.distance(series_arrays[i], series_arrays[j])
            distance_matrix[i][j] = dist
            distance_matrix[j][i] = dist
    print(f"✅ Matriz DTW calculada (método manual): {distance_matrix.shape}")

# Estadísticas de distancias
print(f"\n=== ESTADÍSTICAS DISTANCIAS DTW ===")
print(f"Distancia mínima: {distance_matrix[distance_matrix > 0].min():.3f}")
print(f"Distancia máxima: {distance_matrix.max():.3f}")
print(f"Distancia promedio: {distance_matrix[distance_matrix > 0].mean():.3f}")
print(f"Desviación estándar: {distance_matrix[distance_matrix > 0].std():.3f}")

In [ ]:
# 🎯 CLUSTERING BASADO EN DTW
print("=== CLUSTERING DE PRODUCTOS SIMILARES ===")

# Usar clustering aglomerativo con distancias DTW
n_clusters = min(8, len(productos_dtw) // 3)  # Número adaptativo de clusters
clustering = AgglomerativeClustering(
    n_clusters=n_clusters,
    linkage='average',
    metric='precomputed'
)

clusters = clustering.fit_predict(distance_matrix)

# Crear DataFrame con resultados
df_clusters = pd.DataFrame({
    'product_id': productos_dtw,
    'cluster': clusters
})

print(f"Productos agrupados en {n_clusters} clusters:")
cluster_counts = df_clusters['cluster'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    print(f"  Cluster {cluster_id}: {count} productos")

# Encontrar productos más similares para cada producto
print(f"\n=== PRODUCTOS MÁS SIMILARES (TOP 3) ===")
df_similares = []

for i, product_id in enumerate(productos_dtw):
    # Obtener distancias de este producto a todos los demás
    distancias = distance_matrix[i].copy()
    distancias[i] = np.inf  # Excluir el producto mismo
    
    # Encontrar los 3 más similares (menor distancia)
    indices_similares = np.argsort(distancias)[:3]
    
    for rank, idx in enumerate(indices_similares, 1):
        df_similares.append({
            'product_id': product_id,
            'similar_product': productos_dtw[idx],
            'dtw_distance': distancias[idx],
            'similarity_rank': rank,
            'cluster_origen': clusters[i],
            'cluster_similar': clusters[idx]
        })

df_similares = pd.DataFrame(df_similares)

# Mostrar algunos ejemplos
print("Ejemplos de productos similares:")
for product in productos_dtw[:5]:
    similares = df_similares[df_similares['product_id'] == product]
    print(f"\nProducto {product}:")
    for _, row in similares.iterrows():
        print(f"  #{row['similarity_rank']}: Producto {row['similar_product']} "
              f"(distancia: {row['dtw_distance']:.3f}, cluster: {row['cluster_similar']})")

In [ ]:
# 🔮 PREDICCIÓN DTW - Usar productos similares como templates
print("=== PREDICCIÓN USANDO DTW ===")

# Crear predicciones basadas en productos similares
df_pred_dtw = []

# Para cada producto, usar los similares como referencia
for product_id in productos_dtw:
    # Obtener productos similares
    similares = df_similares[df_similares['product_id'] == product_id].copy()
    
    # Obtener predicción AutoGluon del producto (si existe)
    autogluon_pred = resultado[resultado['product_id'] == product_id]['tn'].iloc[0] if product_id in resultado['product_id'].values else None
    
    # Obtener última observación del producto
    ultimo_valor = ts_matrix.loc[product_id].iloc[-1] if product_id in ts_matrix.index else 0
    
    # MÉTODO 1: Promedio ponderado por similitud (inverso de distancia)
    pred_weighted = 0
    peso_total = 0
    predicciones_similares = []
    
    for _, sim in similares.iterrows():
        similar_id = sim['similar_product']
        distancia = sim['dtw_distance']
        
        # Obtener predicción del producto similar (puede ser AutoGluon o último valor)
        if similar_id in resultado['product_id'].values:
            pred_similar = resultado[resultado['product_id'] == similar_id]['tn'].iloc[0]
        else:
            pred_similar = ts_matrix.loc[similar_id].iloc[-1] if similar_id in ts_matrix.index else 0
        
        # Peso inversamente proporcional a la distancia
        peso = 1 / (1 + distancia)
        pred_weighted += pred_similar * peso
        peso_total += peso
        predicciones_similares.append(pred_similar)
    
    if peso_total > 0:
        pred_weighted /= peso_total
    else:
        pred_weighted = ultimo_valor
    
    # MÉTODO 2: Promedio simple de similares
    pred_avg_similar = np.mean(predicciones_similares) if predicciones_similares else ultimo_valor
    
    # MÉTODO 3: Usar patrón del más similar
    pred_best_similar = predicciones_similares[0] if predicciones_similares else ultimo_valor
    
    # MÉTODO 4: Ensemble DTW + AutoGluon
    if autogluon_pred is not None:
        pred_ensemble = 0.6 * autogluon_pred + 0.4 * pred_weighted
    else:
        pred_ensemble = pred_weighted
    
    df_pred_dtw.append({
        'product_id': product_id,
        'pred_dtw_weighted': pred_weighted,
        'pred_dtw_avg_similar': pred_avg_similar,
        'pred_dtw_best_similar': pred_best_similar,
        'pred_dtw_ensemble': pred_ensemble,
        'autogluon_original': autogluon_pred,
        'ultimo_valor': ultimo_valor,
        'num_similares': len(similares),
        'cluster': df_clusters[df_clusters['product_id'] == product_id]['cluster'].iloc[0]
    })

df_pred_dtw = pd.DataFrame(df_pred_dtw)

print("=== PREDICCIONES DTW GENERADAS ===")
print(df_pred_dtw[['product_id', 'pred_dtw_weighted', 'pred_dtw_ensemble', 
                   'autogluon_original', 'num_similares']].head())

print(f"\n=== ESTADÍSTICAS PREDICCIONES DTW ===")
print(f"Predicción DTW ponderada - Media: {df_pred_dtw['pred_dtw_weighted'].mean():.2f}")
print(f"Predicción DTW ensemble - Media: {df_pred_dtw['pred_dtw_ensemble'].mean():.2f}")
print(f"Correlación DTW vs AutoGluon: {df_pred_dtw['pred_dtw_weighted'].corr(df_pred_dtw['autogluon_original']):.3f}")

# Casos donde DTW difiere significativamente de AutoGluon
df_pred_dtw['diff_dtw_autogluon'] = abs(df_pred_dtw['pred_dtw_weighted'] - df_pred_dtw['autogluon_original'])
print(f"\nDiferencia promedio DTW vs AutoGluon: {df_pred_dtw['diff_dtw_autogluon'].mean():.2f}")

top_diff = df_pred_dtw.nlargest(3, 'diff_dtw_autogluon')
print(f"\nProductos con mayor diferencia DTW vs AutoGluon:")
for _, row in top_diff.iterrows():
    print(f"  Producto {row['product_id']}: DTW={row['pred_dtw_weighted']:.2f}, "
          f"AutoGluon={row['autogluon_original']:.2f}, Diff={row['diff_dtw_autogluon']:.2f}")

In [ ]:
# 📊 CREAR FEATURES DTW PARA MODELO FINAL
print("=== CREANDO FEATURES DTW ===")

# Combinar predicciones DTW con datos originales
df_features_dtw = resultado[['product_id', 'tn']].copy()
df_features_dtw = df_features_dtw.rename(columns={'tn': 'autogluon_pred'})

# Merge con predicciones DTW
df_features_dtw = df_features_dtw.merge(
    df_pred_dtw[['product_id', 'pred_dtw_weighted', 'pred_dtw_ensemble', 
                 'num_similares', 'cluster']], 
    on='product_id', how='left'
)

# Crear features adicionales basadas en DTW
df_features_dtw['dtw_vs_autogluon'] = df_features_dtw['pred_dtw_weighted'] / df_features_dtw['autogluon_pred']
df_features_dtw['dtw_autogluon_diff'] = abs(df_features_dtw['pred_dtw_weighted'] - df_features_dtw['autogluon_pred'])
df_features_dtw['dtw_consensus'] = abs(df_features_dtw['dtw_vs_autogluon'] - 1) < 0.1  # Consenso si diff < 10%

# Feature de confianza basada en número de similares y cluster
df_features_dtw['dtw_confidence'] = np.minimum(df_features_dtw['num_similares'] / 3, 1.0)

# Feature de estabilidad del cluster
cluster_stability = df_features_dtw.groupby('cluster')['dtw_vs_autogluon'].std().reset_index()
cluster_stability.columns = ['cluster', 'cluster_stability']
df_features_dtw = df_features_dtw.merge(cluster_stability, on='cluster', how='left')

# Feature de ranking DTW dentro del cluster  
df_features_dtw['dtw_rank_in_cluster'] = df_features_dtw.groupby('cluster')['pred_dtw_weighted'].rank(ascending=False)

print("=== FEATURES DTW CREADAS ===")
print(df_features_dtw[['product_id', 'autogluon_pred', 'pred_dtw_weighted', 
                       'dtw_vs_autogluon', 'dtw_confidence', 'dtw_consensus']].head())

print(f"\n=== ESTADÍSTICAS FEATURES DTW ===")
print(f"Productos con consenso DTW-AutoGluon: {df_features_dtw['dtw_consensus'].sum()}")
print(f"Confianza promedio DTW: {df_features_dtw['dtw_confidence'].mean():.3f}")
print(f"Estabilidad promedio de clusters: {df_features_dtw['cluster_stability'].mean():.3f}")

# EXPORTAR RESULTADOS DTW
output_dir = "C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/AutoGluon"
os.makedirs(output_dir, exist_ok=True)

# Exportar predicciones DTW
df_features_dtw.to_csv(os.path.join(output_dir, "features_dtw.csv"), index=False)
df_similares.to_csv(os.path.join(output_dir, "productos_similares_dtw.csv"), index=False)
df_clusters.to_csv(os.path.join(output_dir, "clusters_dtw.csv"), index=False)

# Exportar predicción final DTW (formato estándar)
df_export_dtw = df_features_dtw[['product_id', 'pred_dtw_ensemble']].copy()
df_export_dtw = df_export_dtw.rename(columns={'pred_dtw_ensemble': 'tn'})
df_export_dtw.to_csv(os.path.join(output_dir, "predicciones_202002_dtw.csv"), index=False)

print(f"\n=== ARCHIVOS DTW EXPORTADOS ===")
print(f"✅ features_dtw.csv - Features DTW para modelos")
print(f"✅ productos_similares_dtw.csv - Mapa de productos similares")
print(f"✅ clusters_dtw.csv - Clusters de productos")
print(f"✅ predicciones_202002_dtw.csv - Predicciones finales DTW")

print(f"\n🎯 VENTAJAS DTW IDENTIFICADAS:")
print(f"1. 📈 Productos con patrones similares pero desalineados temporalmente")
print(f"2. 🎲 Diversificación de predicciones (correlación con AutoGluon: {df_features_dtw['dtw_vs_autogluon'].mean():.3f})")
print(f"3. 🛡️  Robustez - {df_features_dtw['dtw_consensus'].sum()} productos con consenso")
print(f"4. 🔍 Detección de outliers - {(abs(df_features_dtw['dtw_vs_autogluon'] - 1) > 0.3).sum()} productos con diferencias >30%")
print(f"5. 📊 Clustering automático en {n_clusters} grupos de comportamiento similar")

print(f"\n💡 RECOMENDACIÓN DE USO:")
print(f"- Usar 'pred_dtw_ensemble' como feature adicional en modelos finales")
print(f"- 'dtw_confidence' como peso de confianza")
print(f"- 'dtw_consensus' para identificar predicciones robustas")
print(f"- Clusters DTW para segmentación de productos")